# Colorimetry

Color differences, XYZ/Lab/LCh/xyY conversions, white points, chromatic adaptation and CIECAM02.

In [1]:
import numpy as np

import lcms2
from lcms2 import colorimetry as cm

`delta_e(lab1, lab2, method=...)` measures color differences. It is vectorized over `(..., 3)` arrays
and broadcasts. The default is CIEDE2000; `76`, `94`, `cmc` and `bfd` are also available.

In [2]:
pair = ([50.0, 2.6772, -79.7751], [50.0, 0.0, -82.7485])
print("2000", lcms2.delta_e(*pair).round(4))
for method in ("76", "94", "bfd"):
    print(method.ljust(4), lcms2.delta_e(*pair, method=method).round(4))
print("cmc ", lcms2.delta_e(*pair, method="cmc", l=2, c=1).round(4))
print("2000 Kc=2", lcms2.delta_e(*pair, Kc=2).round(4))

2000 2.0425
76   4.0011
94   1.4083
bfd  1.2195
cmc  1.7387
2000 Kc=2 1.7556


One reference against many colors, converted to Lab first.

In [3]:
rng = np.random.default_rng(0)
palette = rng.integers(0, 256, (5, 3), dtype=np.uint8)
lab = lcms2.convert(palette, "sRGB", "Lab")
print(lcms2.delta_e(lab, lab[0]).round(2))

[ 0.   30.71 52.31 56.89 55.23]


XYZ, xyY, Lab and LCh conversions, each with its inverse. `xyz_to_lab` takes an optional white (default D50). Hue in `lab_to_lch` is in degrees.

In [4]:
xyz = np.array([0.2, 0.3, 0.4])
lab = cm.xyz_to_lab(xyz)
print(lab.round(3), cm.lab_to_xyz(lab).round(6))
print(cm.lab_to_lch(lab).round(3), cm.xyz_to_xyy(xyz).round(4))
print(cm.xyz_to_lab(xyz, cm.D65).round(3))
print(cm.D50, cm.D65)

[ 61.654 -38.74  -23.24 ] [0.2 0.3 0.4]
[ 61.654  45.176 210.96 ] [0.2222 0.3333 0.3   ]
[ 61.654 -37.321  -9.353]
[0.9642 1.     0.8249] [0.95047 1.      1.08883]


White points: named illuminants live in `lcms2.WHITE_POINTS` (per observer angle), and color temperature
converts to and from a white point in xyY.

In [5]:
print(lcms2.WHITE_POINTS[2]["D65"])
white = cm.white_point_from_temperature(5000)
print(white.round(4), round(float(cm.temperature_from_white_point(white))))
print(round(float(cm.temperature_from_white_point(cm.D65_xyY))))

{'xyY': (0.31272, 0.32903, 1.0), 'CCT': 6504.0, 'description': 'noon daylight: television, sRGB color space'}
[0.3457 0.3587 1.    ] 5000
6504


`adapt` moves XYZ between white points with Bradford (default) or another adaptation matrix.

In [6]:
d65_white = np.array([0.95047, 1.0, 1.08883])
color = np.array([0.4, 0.3, 0.2])
print(cm.adapt(d65_white, cm.D65, cm.D50).round(4))
for method in ("bradford", "von_kries", "xyz_scaling"):
    print(method.ljust(11), cm.adapt(color, cm.D65, cm.D50, method=method).round(4))
print(cm.adaptation_matrix(cm.D65, cm.D50).round(4))

[0.9642 1.     0.8249]
bradford    [0.416  0.3056 0.1512]
von_kries   [0.4126 0.3009 0.1515]
xyz_scaling [0.4058 0.3    0.1515]
[[ 1.0478  0.0229 -0.0502]
 [ 0.0296  0.9905 -0.0171]
 [-0.0092  0.0151  0.7518]]


CIECAM02 maps XYZ to lightness, chroma and hue for a viewing condition, and back.

In [7]:
model = lcms2.CIECAM02(white_point=d65_white * 100, adapting_luminance=64, background=20, surround="average")
jch = model.forward(np.array([19.01, 20.0, 21.78]))
print(jch.round(3))
print(model.reverse(jch).round(3))

[482.996   3.629 211.358]
[19.01 20.   21.78]
